# Train và so sánh ba hệ thống OCR

**PaddleOCR → VietLPR_v1 → VietLPR_v2**, dùng cùng dataset đã chia.
Train trên GPU laptop; test ONNX FP32 trên CPU, batch 1, 4 thread.

| Model | ACC (%) ↑ | CER (%) ↓ | Speed (ms/ảnh) ↓ | FPS ↑ | Speedup (×) ↑ |
|---|---:|---:|---:|---:|---:|

PaddleOCR là mốc 1×. Chưa có weights train hoàn chỉnh nên chưa có số đo để điền bảng.
Mặc định các thao tác train/export/test đều tắt. Đọc hướng dẫn trong `experiments/README.md` trước khi bật.


## 1. Chọn môi trường và kiểm tra đường dẫn

Chọn kernel `.venv-ocr` (Python 3.11). Paddle train bằng môi trường riêng `.venv-ocr-paddle`.
`.venv` và `main.ipynb` phục vụ chuẩn hoá dữ liệu vẫn giữ nguyên.


In [ ]:
from pathlib import Path
import json, subprocess, sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "experiments/configs").is_dir())
sys.path.insert(0, str(ROOT))
TORCH_PY = ROOT / ".venv-ocr/Scripts/python.exe"
PADDLE_PY = ROOT / ".venv-ocr-paddle/Scripts/python.exe"
print("Workspace:", ROOT)
print("Python notebook:", sys.executable)
print("Python VietLPR:", TORCH_PY.exists())
print("Python Paddle:", PADDLE_PY.exists())

def run(python, *arguments):
    subprocess.run([str(python), *arguments], cwd=ROOT, check=True)

run(TORCH_PY, "-c", "import torch; assert torch.cuda.is_available(); print('Torch GPU:', torch.cuda.get_device_name(0), 'CUDA', torch.version.cuda)")
run(PADDLE_PY, "-c", "import paddle; assert paddle.is_compiled_with_cuda(); paddle.set_device('gpu:0'); x=paddle.ones([2,2]); print('Paddle GPU:', x.place, 'CUDA', paddle.version.cuda())")


## 2. Source và cấu hình ba model

- PaddleOCR: PP-OCRv4 mobile, source chính thức v2.9.1.
- VietLPR_v1: TinyDBNet + TinyCRNN/GRU hai chiều từ repo GitHub đã gửi.
- VietLPR_v2: FastDBNet/RepConv + FastCRNN/attention tách từ notebook gốc.

Mã nguồn ba model nằm trong `third_party/PaddleOCR/`, `third_party/VietLPR_v1/`, `third_party/VietLPR_v2/`.
Hai repo ngoài được khoá commit; kiến trúc v2 lấy từ notebook đã chốt SHA256.
Chỉ model definition được tách từ notebook cũ; không chạy video, YOLO, tải dữ liệu hoặc test cũ.


In [ ]:
from experiments.config import load_config

for name in ("paddleocr", "vietlpr_v1", "vietlpr_v2"):
    cfg = load_config(name)
    print(name, "| input:", cfg["model"], "| output:", cfg["run_dir"])

DOWNLOAD_SOURCES = False
if DOWNLOAD_SOURCES:
    run(TORCH_PY, "-m", "experiments.bootstrap")


## 3. Kiểm tập dữ liệu

Không chia lại split. Model học trên train, chọn checkpoint/ngưỡng trên val; test để đánh giá cuối.
Đầu vào là ảnh biển đã crop nên không cần YOLO.


In [ ]:
from experiments.data import read_split

DATASET = ROOT / "vn-plate-dataset_v2"
for split in ("train", "val", "test"):
    det = read_split(DATASET, split, "det")
    rec = read_split(DATASET, split, "rec")
    print(split, "| ảnh biển:", len(det), "| crop REC:", len(rec))


## 4. Kiểm code trước khi train

Smoke chỉ dùng 4 mẫu/một epoch và thư mục tạm, không phải weights dùng báo cáo.
Đã có kiểm metric chung để không nhầm ACC từng dòng với ACC cả biển hoặc CER trung bình từng ảnh.


In [ ]:
RUN_CHECKS = False
if RUN_CHECKS:
    run(TORCH_PY, "-m", "unittest", "discover", "-s", "experiments/tests", "-v")
    run(TORCH_PY, "-m", "experiments.smoke", "--device", "cuda", "--full-size")
    run(PADDLE_PY, "-m", "experiments.paddle", "smoke", "--device", "gpu")
else:
    print("Bật RUN_CHECKS để chạy kiểm code, không train dài.")


## 5. Train/val VietLPR_v1 và VietLPR_v2

Train trên GPU với `DEVICE="cuda"`; code báo lỗi rõ nếu GPU không hoạt động.
Train từng DET/REC theo thứ tự để tránh thiếu VRAM. Mỗi epoch đánh giá val và lưu best/last.
Config mặc định train từ đầu; không dùng weights không rõ nguồn trong repo v1.

Đổi batch, epoch, patience trong `experiments/configs/*.json`. Để tiếp tục lượt bị ngắt, dùng CLI với `--resume`.


In [ ]:
RUN_TRAIN_VIETLPR = False
DEVICE = "cuda"  # Đổi thành "cpu" nếu đã chấp nhận train chậm hơn.
if RUN_TRAIN_VIETLPR:
    for name in ("vietlpr_v1", "vietlpr_v2"):
        for task in ("det", "rec"):
            run(TORCH_PY, "-m", "experiments.train_torch", "--model", name, "--task", task, "--device", DEVICE)
else:
    print("Chưa chạy train VietLPR.")


## 6. Train/val PaddleOCR

`prepare` xuất YAML từ config chính thức, dùng đường dẫn train/val đã có, bộ ký tự chung và tắt pretrained mặc định.
Paddle dùng môi trường riêng, đã chọn `PADDLE_DEVICE="gpu"` để train trên GPU.


In [ ]:
RUN_TRAIN_PADDLE = False
PADDLE_DEVICE = "gpu"
if RUN_TRAIN_PADDLE:
    run(PADDLE_PY, "-m", "experiments.paddle", "prepare", "--device", PADDLE_DEVICE)
    for task in ("det", "rec"):
        run(PADDLE_PY, "-m", "experiments.paddle", "train", "--task", task)
else:
    print("Chưa chạy train PaddleOCR.")


## 7. Xuất ONNX FP32

Mỗi hệ thống xuất `det.onnx`, `rec.onnx`, `config.json` và kiểm số học framework → ONNX.
V2 gộp nhánh RepConv trước khi xuất. Batch inference cố định 1; các crop trong cùng biển chạy tuần tự.


In [ ]:
RUN_EXPORT = False
if RUN_EXPORT:
    for name in ("vietlpr_v1", "vietlpr_v2"):
        run(TORCH_PY, "-m", "experiments.export", "--model", name)
    for task in ("det", "rec"):
        run(PADDLE_PY, "-m", "experiments.paddle", "export", "--task", task)
else:
    print("Chưa xuất ONNX weights thành phẩm.")


## 8. Chọn ngưỡng trên val và khoá model

`tune` chỉ đọc val. Chọn ACC cao nhất rồi CER thấp nhất; lưu SHA model/config vào `frozen.json`.
Không chỉnh ngưỡng theo kết quả test. Muốn giữ nguyên ngưỡng thì thêm `--no-search` ở CLI.


In [ ]:
RUN_SELECT_VAL = False
if RUN_SELECT_VAL:
    for name in ("paddleocr", "vietlpr_v1", "vietlpr_v2"):
        run(TORCH_PY, "-m", "experiments.evaluate", "tune", "--model", name)
else:
    print("Chưa khoá model để test.")


## 9. Test CPU và tạo bảng

ACC/CER trên chuỗi cả biển. Mẫu không detect được vẫn tính lỗi. Speed gồm preprocess, DET, crop, REC và decode;
không tính đọc ổ đĩa, load model hoặc vẽ ảnh. Dùng cùng runtime CPU, thread, warm-up và số lượt lặp.
Không lấy ACC của model gốc ghép với Speed của model xuất khác.


In [ ]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    for name in ("paddleocr", "vietlpr_v1", "vietlpr_v2"):
        run(TORCH_PY, "-m", "experiments.evaluate", "test", "--model", name)
    run(TORCH_PY, "-m", "experiments.evaluate", "compare")
else:
    print("Chưa chạy test cuối; chưa có bảng hiệu năng chính thức.")


## 10. Đọc kết quả

`runs/comparison.csv`: **Model | ACC | CER | Speed | FPS | Speedup**.
`runs/<model>/test_predictions.json`: nhãn thật, chuỗi dự đoán, thời gian từng ảnh.
`runs/<model>/test_result.json`: kết quả và điều kiện đo. `runs/` được Git bỏ qua.


In [ ]:
import csv

table = ROOT / "runs/comparison.csv"
if table.exists():
    with table.open(encoding="utf-8-sig", newline="") as stream:
        for row in csv.DictReader(stream):
            print(row)
else:
    print("Chưa có kết quả thực tế. Không điền số liệu minh hoạ vào bảng chính.")
